# EXP-028 Stage 1 — Colab CPU

Choose **Runtime → Change runtime type → CPU**, then **Runtime → Run all**. The default runs the authorized exploratory fold-3 comparison. Set `AUDIT_ONLY=True` for cache verification alone.

The notebook uses the existing `MyDrive/exp023_handoff.zip` and original `MyDrive/exp026/features/8f1e57257da52d7212b0/`. It reconstructs the three frozen visual NPZs from the handoff and checks their original SHA-256 values. The complete Qwen cache is read directly from Drive, one NPZ at a time.

After verification, the attempt directory under `MyDrive/exp028_stage1_cpu/<source-prefix>/` contains the results, logs and verified `outputs.zip`. The result table appears in the notebook. A blocked run exports a failure receipt. If the RAM preflight fails, choose a CPU High-RAM runtime and run again.

TASK-035 runs isolated SciPy and scikit-learn L-BFGS smoke checks and the original crash regression before the full tests. Use a fresh CPU session. The one-thread native profile is fixed for the checks and screen. A failed check exports the process receipt and stops.


In [ ]:
# OWNER_CONFIGURATION
AUDIT_ONLY = False
INPUT_ZIP = "/content/drive/MyDrive/exp023_handoff.zip"
QWEN_CACHE = "/content/drive/MyDrive/exp026/features/8f1e57257da52d7212b0"
PERSISTENT_ROOT = "/content/drive/MyDrive/exp028_stage1_cpu"
LOCAL_ROOT = "/content/exp028_stage1_cpu"
SOURCE_REVISION = "bbbbe3c786fc4ec541cd84b9c30f7e824bf387d1"
MAX_TOTAL_WALL_HOURS = 2.0
AUTO_BROWSER_DOWNLOAD = True
AUTO_RELEASE_COLAB = True

# Fixed scope for the existing CPU fold-3 experiment.
AUTHORIZE_EXPERIMENT = True
AUTHORIZE_GUARDED_CONFIRMATION = False
EXECUTION_SCOPE = "RESEARCH_ONLY_CV"
RESOURCE_ELIGIBILITY_STATUS = "RESEARCH_ONLY_COMPETITION_UNCONFIRMED"
MAX_GPU_HOURS = 0
REQUIRE_SUBMISSION_ZIP = False
EVAL_INPUT_CSV = None
EVAL_DATA_DIR = None


In [ ]:
import hashlib
import json
import math
import os
from pathlib import Path
import signal
import subprocess
import sys
import time
import uuid
import zipfile


def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def write_receipt(path, value):
    temp = path.with_suffix(".part")
    temp.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    temp.replace(path)


def invoke(command, log, env, timeout):
    with log.open("w", encoding="utf-8") as output:
        task = subprocess.Popen(command, env=env, stdout=output, stderr=subprocess.STDOUT,
                                start_new_session=True)
        try:
            return task.wait(timeout=timeout)
        except BaseException:
            if task.poll() is None:
                os.killpg(task.pid, signal.SIGKILL)
                task.wait()
            raise


def verified_export(attempt):
    names = [p.relative_to(attempt).as_posix() for p in sorted(attempt.rglob("*"))
             if p.is_file() and not any(t.startswith(".") or t.endswith(".part") for t in p.relative_to(attempt).parts)]
    members = {name: sha256(attempt / name) for name in names}
    write_receipt(attempt / "manifest_sha256.json", members)
    partial = attempt / "outputs.zip.part"
    with zipfile.ZipFile(partial, "w", zipfile.ZIP_DEFLATED) as out:
        for name in (*members, "manifest_sha256.json"):
            out.write(attempt / name, name)
    final = attempt / "outputs.zip"
    partial.replace(final)
    with zipfile.ZipFile(final) as archive:
        archive_names = archive.namelist()
        if len(set(archive_names)) != len(archive_names) or set(archive_names) != {*members, "manifest_sha256.json"}:
            raise OSError("Export ZIP member list differs")
        if json.loads(archive.read("manifest_sha256.json")) != members:
            raise OSError("Export manifest read-back differs")
        for name, expected in members.items():
            with archive.open(name) as handle:
                if hashlib.file_digest(handle, "sha256").hexdigest() != expected:
                    raise OSError(f"Export ZIP checksum differs for {name}")
    return final, sha256(final)


def run_pipeline(settings, drive, files, runtime):
    started = time.monotonic()
    tag = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()) + "-" + uuid.uuid4().hex[:8]
    revision = settings["SOURCE_REVISION"]
    local = Path(settings["LOCAL_ROOT"])
    attempt = Path(settings["PERSISTENT_ROOT"]) / revision[:12] / tag
    source = local / ("source_" + revision[:12])
    result = {"status": "BLOCKED_VERIFICATION", "device": "CPU",
              "source_revision": revision, "run_id": tag, "browser_arrival": "UNVERIFIED"}
    stage = "MOUNT"
    mounted = False
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", PYTHONNOUSERSITE="1",
               OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1", OMP_NUM_THREADS="1",
               OMP_THREAD_LIMIT="1", NUMEXPR_NUM_THREADS="1", BLIS_NUM_THREADS="1",
               ST3_EXP028_CPU_THREADS="1", PYTHONFAULTHANDLER="1",
               UV_PYTHON_INSTALL_DIR=str(local / "python"), UV_CACHE_DIR=str(local / "uv"))
    env.pop("PYTHONPATH", None)

    def remaining(reserve=120):
        available = settings["MAX_TOTAL_WALL_HOURS"] * 3600 - (time.monotonic() - started) - reserve
        if available <= 0:
            raise TimeoutError("CPU execution budget exhausted")
        return available

    def command(args, name, *, allow_failed=False):
        print(f"Running {name}. Logs: {attempt / (name + '.log')}", flush=True)
        code = invoke(args, attempt / (name + ".log"), env, remaining())
        if code and not allow_failed:
            failure = signal.Signals(-code).name if code < 0 else f"exit code {code}"
            raise RuntimeError(f"{name} failed with {failure}. See {name}.log")
        return code

    try:
        drive.mount("/content/drive")
        mounted = True
        attempt.mkdir(parents=True, exist_ok=False)
        stage = "CONFIGURATION"
        if (settings["MAX_GPU_HOURS"] != 0 or settings["AUTHORIZE_GUARDED_CONFIRMATION"]
                or settings["REQUIRE_SUBMISSION_ZIP"] or settings["EVAL_INPUT_CSV"] is not None
                or settings["EVAL_DATA_DIR"] is not None or settings["EXECUTION_SCOPE"] != "RESEARCH_ONLY_CV"
                or settings["RESOURCE_ELIGIBILITY_STATUS"] != "RESEARCH_ONLY_COMPETITION_UNCONFIRMED"):
            raise ValueError("Select the frozen CPU research scope in the notebook")
        budget = settings["MAX_TOTAL_WALL_HOURS"]
        if not math.isfinite(budget) or not 0 < budget <= 3:
            raise ValueError("CPU allocation must be finite and at most 3 hours")
        if Path("/dev/nvidia0").exists() or os.environ.get("COLAB_GPU", "0") not in ("", "0"):
            raise ValueError("Select CPU in Runtime → Change runtime type, then Run all")
        stage = "DRIVE_INPUTS"
        if not Path(settings["INPUT_ZIP"]).is_file():
            raise FileNotFoundError(f"Expected the unchanged handoff ZIP at {settings['INPUT_ZIP']}")
        if not Path(settings["QWEN_CACHE"]).is_dir():
            raise FileNotFoundError(f"Expected the original Qwen cache directory at {settings['QWEN_CACHE']}")
        local.mkdir(parents=True, exist_ok=True)
        stage = "SOURCE"
        if not source.exists():
            repo = "https://github.com/bqa100507-spec/SemEval-2027-Task-12---ArtEmotion-28-Subtask-3---Image-Only-Cross-Cultural-Sentiment-Prediction.git"
            command(["git", "clone", "--no-checkout", repo, str(source)], "clone")
            command(["git", "-C", str(source), "checkout", "--detach", revision], "checkout")
        check_git = ["git", "-C", str(source)]
        actual = subprocess.check_output([*check_git, "rev-parse", "HEAD"], text=True, timeout=15).strip()
        dirty = subprocess.check_output([*check_git, "status", "--porcelain", "--untracked-files=no"],
                                        text=True, timeout=15)
        if actual != revision or dirty:
            raise ValueError("Expected a clean checkout at the exact pinned source revision")
        stage = "DEPENDENCIES"
        command([sys.executable, "-m", "pip", "install", "uv==0.7.13"], "uv")
        uv = [sys.executable, "-m", "uv"]
        command([*uv, "python", "install", "3.12.10"], "python")
        venv = local / ("venv_" + revision[:12])
        python = str(venv / "bin/python")
        if not Path(python).is_file():
            command([*uv, "venv", "--python", "3.12.10", str(venv)], "venv")
        command([*uv, "pip", "sync", "--python", python, str(source / "requirements-exp028-cpu.lock.txt")], "dependencies")
        command([*uv, "pip", "check", "--python", python], "dependency_check")
        stage = "NATIVE_PREFLIGHT"
        native = attempt / "native"
        command([python, "-u", str(source / "scripts/exp028_native_preflight.py"),
                 "--output", str(native)], "native_preflight")
        gate = json.loads((native / "native_preflight.json").read_text(encoding="utf-8"))
        if (gate["status"] != "PASS" or set(gate["checks"]) != {"scipy", "sklearn", "original_regression"}
                or any(check["status"] != "PASS" or check["exit_code"] != 0 for check in gate["checks"].values())):
            raise ValueError("Native preflight did not verify all three subprocesses")
        stage = "CPU_TESTS"
        command([python, "-m", "pytest", "-q", str(source / "tests/test_exp028_stage1.py"),
                 str(source / "tests/test_exp028_colab_cpu.py"),
                 str(source / "tests/test_experiment_workflow.py")], "cpu_tests")
        stage = "CPU_WORKFLOW"
        seconds = int(remaining(reserve=180))
        if seconds <= 0:
            raise TimeoutError("No time remains for the CPU worker")
        audit_only = settings["AUDIT_ONLY"] or not settings["AUTHORIZE_EXPERIMENT"]
        worker = [python, "-u", str(source / "scripts/run_exp028_colab_cpu.py"),
                  "--handoff", settings["INPUT_ZIP"], "--qwen-cache", settings["QWEN_CACHE"],
                  "--scratch", str(local / tag), "--attempt", str(attempt),
                  "--revision", revision, "--seconds", str(seconds)]
        exit_code = command(worker + (["--audit-only"] if audit_only else []), "cpu_workflow", allow_failed=True)
        path = attempt / "workflow_status.json"
        if not path.is_file():
            raise RuntimeError(f"CPU workflow exited {exit_code} without a status receipt. See cpu_workflow.log")
        report = json.loads(path.read_text(encoding="utf-8"))
        if exit_code:
            stage = report.get("failed_stage", stage)
            raise RuntimeError(report.get("error", f"CPU workflow exited {exit_code}"))
        expected = "COMPLETE_CACHE_VERIFIED" if audit_only else "FOLD3_SCREEN_COMPLETE / EXPLORATORY"
        if report["status"] != expected or report["source_revision"] != revision:
            raise ValueError("Worker did not produce the expected verified receipt")
        result["status"] = expected
    except Exception as error:
        result.update(status="BLOCKED_VERIFICATION", failed_stage=stage, error=f"{type(error).__name__}: {error}")
    finally:
        try:
            if mounted:
                result["elapsed_before_export_seconds"] = time.monotonic() - started
                write_receipt(attempt / "status.json", result)
                output, digest = verified_export(attempt)
                result["export_readback"] = "PASS"
                result["export_zip_sha256"] = digest
                if result["status"] == "FOLD3_SCREEN_COMPLETE / EXPLORATORY":
                    from IPython.display import display, Markdown
                    display(Markdown((attempt / "summary.md").read_text(encoding="utf-8")))
                if settings["AUTO_BROWSER_DOWNLOAD"]:
                    result["download_request"] = "REQUESTED"
                    write_receipt(attempt / "download_attempt.json", result)
                    try:
                        files.download(str(output))
                        result["download_request"] = "API_RETURNED"
                    except Exception as error:
                        result["download_request"] = f"FAILED: {error}"
        except Exception as error:
            result.update(status="BLOCKED_VERIFICATION", failed_stage="EXPORT", export_readback="FAIL",
                          error=f"{type(error).__name__}: {error}")
        finally:
            result["elapsed_seconds"] = time.monotonic() - started
            result["runtime_release"] = "REQUESTED_NOT_CONFIRMED" if settings["AUTO_RELEASE_COLAB"] else "DISABLED"
            try:
                if mounted:
                    write_receipt(attempt / "final_report.json", result)
                print(json.dumps(result, indent=2))
            finally:
                if settings["AUTO_RELEASE_COLAB"]:
                    try:
                        runtime.unassign()
                    except Exception as error:
                        result["runtime_release"] = f"FAILED: {error}"
                        if mounted:
                            write_receipt(attempt / "final_report.json", result)
                        print(result["runtime_release"])
    return result


In [ ]:
from google.colab import drive, files, runtime

run_cloud = run_pipeline
result = run_cloud(globals(), drive, files, runtime)
